## Description
This script organizes a list of menus and submenus from the "Activity Name" column in the CAR data into a hierarchical structure (Tier1–Tier3) to support hierarchical filtering and analysis in Power BI. It also groups specific single-level menus under a new Tier1 category called "Miscellaneous" to make filters easier to sort through. The resulting CSV file should be uploaded into Power BI and linked with your CAR data files via the "Activity Name" column to enable hierarchical filtering (when you do this, make sure the cross-filtering setting is set to "Both").

Note that this is an updated version of my previous code for hierarchical filtering, which includes a cleaner and more user-friendly setup for the filter box in Power BI. The CSV file created by this code is something that I manually created based on the menu maps to better meet business needs.

In [4]:
import pandas as pd
import numpy as np
from io import StringIO

# --- Step 1: Paste raw table (tab-separated, WITHOUT Tier column) ---
raw = """Activity_Name	Tier1	Tier2	Tier3
CCB	Miscellaneous	CCB	
ClosedQueueMenu	Closed Queue Menu		
ClinicVoicemailTransfer	Closed Queue Menu	Clinic Voicemail Transfer	
DisconnectContact	Disconnect Contact		
DisconnectContact1	Disconnect Contact	Disconnect Contact 1	
DisconnectContact2	Disconnect Contact	Disconnect Contact 2	
DisconnectCallbackContact	Disconnect Contact	Disconnect Callback Contact	
FarmworkerMainMenu	Farmworker Main Menu		
FrontDeskTransfer	Front Desk Transfer		
FrontDeskTransfer1	Front Desk Transfer	Front Desk Transfer 1	
FrontDeskTransfer2	Front Desk Transfer	Front Desk Transfer 2	
FrontDeskTransfer3	Front Desk Transfer	Front Desk Transfer 3	
IntakePreQueueMessage1	Miscellaneous	Intake Pre-QueueMessage 1	
PreQueueMessage2	Miscellaneous	Intake Pre-QueueMessage 1	Pre-QueueMessage 2
LegalServerScreenPop	Miscellaneous	Intake Pre-QueueMessage 1	Legal Server Screen-Pop
ScreenPopProcessComplete	Miscellaneous	Intake Pre-QueueMessage 1	Screen-Pop: Process Complete
PlayMOH300s	Miscellaneous	Intake Pre-QueueMessage 1	Play Music
QueueMenu1	Miscellaneous	Intake Pre-QueueMessage 1	Queue Menu 1
LegalMenu2	Legal Issues		
ADAPTQueue	Legal Issues	ADAPT Queue	
ADAPTSPQueue	Legal Issues	ADAPT Queue	
CriminalRecordsVoicemailTransfer	Legal Issues	Criminal Records Voicemail Transfer	
ConsumerSPQueue	Legal Issues	Consumer Queue	
ConsumerQueue	Legal Issues	Consumer Queue	
HousingMenu	Legal Issues	Housing Menu	
TenantMenu	Legal Issues	Housing Menu	TenantMenu
TenantDeterrenceMenu	Legal Issues	Housing Menu	TenantDeterrenceMenu
HousingQueue	Legal Issues	Housing Menu	Housing Queue
HousingSPQueue	Legal Issues	Housing Menu	Housing Queue
EmploymentMenu	Legal Issues	Employment Menu	
EmploymentQueue	Legal Issues	Employment Menu	Employment Queue
EmploymentSPQueue	Legal Issues	Employment Menu	Employment Queue
WorkersCompMenu	Legal Issues	Employment Menu	Workers Comp Menu
ImmigrationMenu	Legal Issues	Employment Menu	
ImmigrationOtherMenu	Legal Issues	Employment Menu	Immigration Other Menu
ImmigrationSPQueue	Legal Issues	Employment Menu	Immigration Queue
ImmigrationQueue	Legal Issues	Employment Menu	Immigration Queue
MigrantVoicemailTransfer	Legal Issues	Employment Menu	Migrant Voicemail Transfer
TraffickingVoicemailTransfer	Legal Issues	Employment Menu	Trafficking Voicemail Transfer
FamilyMenu	Legal Issues	Family Menu	
SimpleDivorceMenu	Legal Issues	Family Menu	Simple Divorce Menu
ChildSupportMenu	Legal Issues	Family Menu	Child Support Menu
FamilyQueue	Legal Issues	Family Menu	Family Queue
EducationQueue	Legal Issues	Family Menu	Education Queue
EducationSPQueue	Legal Issues	Family Menu	Education Queue
FamilySPQueue	Legal Issues	Family Menu	Family Queue
OtherLegalMenu	Legal Issues	Other Legal Menu	
OtherLegalCriminalCaseMenu	Legal Issues	Other Legal Menu	Other Legal Criminal Case Menu
OtherLegalPersonalInjuryMenu	Legal Issues	Other Legal Menu	Other Legal Personal Injury Menu
OtherLegalOtherMenu	Legal Issues	Other Legal Menu	Other Legal Other Menu
BenefitsMenu	Legal Issues	Benefits Menu	
BenefitsQueue	Legal Issues	Benefits Menu	Benefits Queue
BenefitsSPQueue	Legal Issues	Benefits Menu	Benefits Queue
VeteransBenefitsVoicemailTransfer	Legal Issues	Benefits Menu	Veterans Benefits Voicemai lTransfer
HIVMenu	Legal Issues	HIV Menu	
HIVVoicemailTransfer	Legal Issues	HIV Menu	HIV Voicemail Transfer
MainMenu	Main Menu		
AddressFaxHoursMenu	Main Menu	Address Fax Hours Menu	
ComplimentOrComplaintMenu	Main Menu	Compliment Or Complaint Menu	
AppointmentMenu	Main Menu	Appointment Menu	
HolidayPrompt	Main Menu	Holiday Prompt	
LanguageSelectionMenu	Main Menu	Language Selection Menu	
StaffDirectoryEnglishTransfer	Main Menu	Staff Directory English Transfer	
StaffDirectorySpanishTransfer	Main Menu	Staff Directory  Spanish Transfer	
SetCallerID	Miscellaneous	Set Caller ID	
HelpWithLegalorOtherReasonMenu	Miscellaneous	Help With Legal or Other Reason Menu	
ClosedMenu	Miscellaneous	Closed Menu	
ReadANI	Miscellaneous	ReadANI	
CallbackRetry	Miscellaneous	CallbackRetry	
ConfirmCallbackNumber	Miscellaneous	Confirm Callback Number	
CollectCallbackNumber	Miscellaneous	Collect Callback Number	
TransferToSafeHaven	Miscellaneous	Transfer To Safe Haven	
SuburbsOrCityMenu	Seniors Menu		
SeniorNotCookCoMenu	Seniors Menu	Seniors: Not Cook Count Menu (exit)	
SuburbanSeniorsMenu	Seniors Menu	Suburban Seniors Menu	
SubSeniorHomeownerQueue	Seniors Menu	Suburban Seniors Menu	SS Homeowner Queue
SubSeniorHomeownerSPQueue	Seniors Menu	Suburban Seniors Menu	SS Homeowner Queue
SubSeniorOtherQueue	Seniors Menu	Suburban Seniors Menu	SS Other Queue
SubSeniorPreQueueMessage1_1	Seniors Menu	Suburban Seniors Menu	SS Pre-Queue Message
SubSeniorConsumerQueue	Seniors Menu	Suburban Seniors Menu	SS Consumer Queue
SubSeniorTenantQueue	Seniors Menu	Suburban Seniors Menu	SS Tenant Queue
SubSeniorPreQueueMessage1_2	Seniors Menu	Suburban Seniors Menu	SS Pre-Queue Message
BenefitsSubSeniorsQueue	Seniors Menu	Suburban Seniors Menu	SS Benefits Queue
SubSeniorFamilyQueue	Seniors Menu	Suburban Seniors Menu	SS Family Queue
SubSeniorBenefitsQueue	Seniors Menu	Suburban Seniors Menu	SS Benefits Queue
SubSeniorTenantSPQueue	Seniors Menu	Suburban Seniors Menu	SS Tenant Queue
SubSeniorEmploymentQueue	Seniors Menu	Suburban Seniors Menu	SS Employment Queue
SubSeniorConsumerSPQueue	Seniors Menu	Suburban Seniors Menu	SS Consumer Queue
HousingSubSeniorsQueue	Seniors Menu	Suburban Seniors Menu	SS Housing Queue
EmploymentSubSeniorsSPQueue	Seniors Menu	Suburban Seniors Menu	SS Employment Queue
OtherSubSeniorsQueue	Seniors Menu	Suburban Seniors Menu	SS Other Queue
FamilySubSeniorsSPQueue	Seniors Menu	Suburban Seniors Menu	SS Family Queue
HousingSubSeniorsSPQueue	Seniors Menu	Suburban Seniors Menu	SS Housing Queue
BenefitsSubSeniorsSPQueue	Seniors Menu	Suburban Seniors Menu	SS Benefits Queue
ConsumerSubSeniorsQueue	Seniors Menu	Suburban Seniors Menu	SS Consumer Queue
EmploymentSubSeniorsQueue	Seniors Menu	Suburban Seniors Menu	SS Employment Queue
OtherSubSeniorsSPQueue	Seniors Menu	Suburban Seniors Menu	SS Other Queue
FamilySubSeniorsQueue	Seniors Menu	Suburban Seniors Menu	SS Family Queue
HomeownerSubSeniorsQueue	Seniors Menu	Suburban Seniors Menu	SS Homeowner Queue
ConsumerSubSeniorsSPQueue	Seniors Menu	Suburban Seniors Menu	SS Consumer Queue
SubSeniorEmploymentSPQueue	Seniors Menu	Suburban Seniors Menu	SS Employment Queue
SubSeniorOtherSPQueue	Seniors Menu	Suburban Seniors Menu	SS Other Queue
SubSeniorFamilySPQueue	Seniors Menu	Suburban Seniors Menu	SS Family Queue
SubSeniorBenefitsSPQueue	Seniors Menu	Suburban Seniors Menu	SS Benefits Queue
SeniorsADAPTMenu	Seniors Menu	City Seniors (ADAPT Menu)	
SubSeniorADAPTQueue	Seniors Menu	Suburban Seniors Menu	SS ADAPT Queue
ADAPTSubSeniorsQueue	Seniors Menu	Suburban Seniors Menu	SS ADAPT Queue
SubSeniorADAPTSPQueue	Seniors Menu	Suburban Seniors Menu	SS ADAPT Queue
ADAPTSubSeniorsSPQueue	Seniors Menu	Suburban Seniors Menu	SS ADAPT Queue
"""

# --- Step 2: Read the string into a DataFrame ---
df = pd.read_csv(StringIO(raw), sep="\t", dtype=str)

# --- Step 3: Fill missing columns & clean ---
for col in ["Activity_Name", "Tier1", "Tier2", "Tier3"]:
    if col not in df.columns:
        df[col] = np.nan

# Replace empty or whitespace-only strings with NaN
df = df.replace(r'^\s*$', np.nan, regex=True)

# --- Step 4: Save to CSV for Power BI ---
df.to_csv("Hierarchical_Tiers_fixed.csv", index=False)

# --- Step 5: Preview ---
display(df.head(10))


,Activity_Name,Tier1,Tier2,Tier3
0,CCB,Miscellaneous,CCB,NaN
1,ClosedQueueMenu,Closed Queue Menu,NaN,NaN
2,ClinicVoicemailTransfer,Closed Queue Menu,Clinic Voicemail Transfer,NaN
3,DisconnectContact,Disconnect Contact,NaN,NaN
4,DisconnectContact1,Disconnect Contact,Disconnect Contact 1,NaN
5,DisconnectContact2,Disconnect Contact,Disconnect Contact 2,NaN
6,DisconnectCallbackContact,Disconnect Contact,Disconnect Callback Contact,NaN
7,FarmworkerMainMenu,Farmworker Main Menu,NaN,NaN
8,FrontDeskTransfer,Front Desk Transfer,NaN,NaN
9,FrontDeskTransfer1,Front Desk Transfer,Front Desk Transfer 1,NaN
